In [88]:
from collections import deque
import spot

class LTLMutator:
    def __init__(self, ap_list):
        self.ap_set = set(ap_list)
        self.aps = [spot.formula.ap(p) for p in self.ap_set]
        
        self.unary_ops = {
            spot.op_Not: spot.formula.Not,
            spot.op_X: spot.formula.X,
            spot.op_F: spot.formula.F,
            spot.op_G: spot.formula.G
        }
        
        self.binary_ops = {
            spot.op_U: spot.formula.U,
            spot.op_R: spot.formula.R,
            spot.op_W: spot.formula.W,
            spot.op_And: lambda a, b: spot.formula.And([a, b]),
            spot.op_Or: lambda a, b: spot.formula.Or([a, b])
        }
        
        self.rule3d_ops = {
            'U': spot.formula.U,
            'W': spot.formula.W,
            '&': lambda a, b: spot.formula.And([a, b]),
            '|': lambda a, b: spot.formula.Or([a, b])
        }

    def mutate(self, phi):
        if isinstance(phi, str):
            phi = spot.formula(phi)
            
        phi = phi.unabbreviate("ie")
        mutations = {}
        
        for mut_f in self._mutate_recursive(phi):
            s = mut_f.to_str()
            if s not in mutations:
                mutations[s] = mut_f
                
        orig_s = phi.to_str()
        if orig_s in mutations:
            del mutations[orig_s]
            
        return list(mutations.values())

                    
    def _mutate_recursive(self, f):
        # --- GENERAL CASES (Strictly restricted based on your structural goal) ---
        # Rule 5: Wrap the current sub-formula in a unary operator
        for op_func in self.unary_ops.values():
            yield op_func(f)
            
        kind = f.kind()
        
        # --- BASE CASES / LEAF MUTATIONS ---
        # Rule 1 & Rule 6 (Moved here so they only mutate leaves, not the whole tree)
        # if f.is_tt():
        #     yield spot.formula.ff()
            
        # if f.is_ff():
        #     yield spot.formula.tt()
            

        if f in self.ap_set:
            # Rule 2: Swap APs
            for p in self.aps:
                if p != f:
                    yield p
            # Rule 6 for APs: allow mapping an AP to other APs (handled above)
        
        # If we are at the top level of a complex formula, we block Rule 6 
        # from replacing the entire tree with a single 'p' or 'true'.
        # if not is_top_level or f in self.ap_set or f.is_tt() or f.is_ff():
        #     for p in self.aps:
        #         yield p

        # --- INDUCTIVE CASES ---
        if kind in self.unary_ops:
            child = f[0]
            
            # 3(a) Change unary operator
            for other_kind, op_func in self.unary_ops.items():
                if other_kind != kind:
                    yield op_func(child)
                    
            # 3(b) Drop operator
            yield child
            
            # 3(c) Mutate child
            for mutated_child in self._mutate_recursive(child):
                yield self.unary_ops[kind](mutated_child)
                
            # 3(d) Append binary operator
            for p in self.aps:
                for op_func in self.rule3d_ops.values():
                    yield op_func(p, f)

            # 3(e) Swap adjacent unary operators: Op1(Op2(x)) -> Op2(Op1(x))
            if child.kind() in self.unary_ops:
                grandchild = child[0]
                outer_func = self.unary_ops[kind]
                inner_func = self.unary_ops[child.kind()]
                yield inner_func(outer_func(grandchild))
       
                    
        elif kind in self.binary_ops:
            children = list(f)
            
            if len(children) >= 2:
                phi_1 = children[0]
                phi_2 = children[1]
                if len(children) > 2:
                    if kind == spot.op_And:
                        phi_2 = spot.formula.And(children[1:])
                    elif kind == spot.op_Or:
                        phi_2 = spot.formula.Or(children[1:])
                    
                    
                # 4(a) Change binary operator
                for other_kind, op_func in self.binary_ops.items():
                    if other_kind != kind:
                        yield op_func(phi_1, phi_2)
                        
                # 4(b) Keep one child
                yield phi_1
                yield phi_2
                
                # 4(c) Mutate left child 
                for mutated_left in self._mutate_recursive(phi_1):
                    yield self.binary_ops[kind](mutated_left, phi_2)
                    
                # 4(d) Mutate right child 
                for mutated_right in self._mutate_recursive(phi_2):
                    yield self.binary_ops[kind](phi_1, mutated_right)

        return float('inf'), [] # If no structural mutation path connects them


# class LTLDistanceCalculator:
#     def __init__(self, mutator):
#         self.mutator = mutator
#     def _formula_size(self, f):
#         if f.size() == 0: return 1
#         return 1 + sum(self._formula_size(f[i]) for i in range(f.size()))

#     def calculate_raw_distance(self, phi_a, phi_b):
#         phi_a, phi_b = spot.formula(phi_a), spot.formula(phi_b)
#         if phi_a.to_str() == phi_b.to_str():
#             return 0

#         # Queues and Visited sets for both directions
#         # Stores: (formula_object, distance_from_start)
#         queue_f = deque([(phi_a, 0)])
#         visited_f = {phi_a.to_str(): 0}
        
#         queue_b = deque([(phi_b, 0)])
#         visited_b = {phi_b.to_str(): 0}

#         while queue_f and queue_b:
#             # Expand forward
#             curr_f, dist_f = queue_f.popleft()
#             for neighbor in self.mutator.mutate(curr_f):
#                 n_str = neighbor.to_str()
#                 if n_str in visited_b:
#                     return dist_f + 1 + visited_b[n_str]
#                 if n_str not in visited_f:
#                     visited_f[n_str] = dist_f + 1
#                     queue_f.append((neighbor, dist_f + 1))

#             # Expand backward
#             curr_b, dist_b = queue_b.popleft()
#             for neighbor in self.mutator.mutate(curr_b):
#                 n_str = neighbor.to_str()
#                 if n_str in visited_f:
#                     return dist_b + 1 + visited_f[n_str]
#                 if n_str not in visited_b:
#                     visited_b[n_str] = dist_b + 1
#                     queue_b.append((neighbor, dist_b + 1))
        
#         return float('inf')

#     def calculate_normalized_distance(self, phi_a_str, phi_b_str):
#         phi_a = spot.formula(phi_a_str)
#         phi_b = spot.formula(phi_b_str)
#         raw_dist = self.calculate_raw_distance(phi_a, phi_b)
        
#         if raw_dist == float('inf'): return float('inf')
#         max_size = max(self._formula_size(phi_a), self._formula_size(phi_b))
#         return raw_dist / max_size if max_size > 0 else 0.0


class LTLDistanceCalculator:
    def __init__(self, mutator):
        self.mutator = mutator

    def _formula_size(self, f):
        if f.size() == 0: return 1
        return 1 + sum(self._formula_size(f[i]) for i in range(f.size()))

    def calculate_raw_distance(self, phi_a, phi_b):
        phi_a, phi_b = spot.formula(phi_a), spot.formula(phi_b)
        if phi_a.to_str() == phi_b.to_str():
            return 0, [phi_a.to_str()]

        # Queues store: (formula_object, distance_from_start)
        queue_f = deque([(phi_a, 0)])
        queue_b = deque([(phi_b, 0)])
        
        # Visited dicts store: { formula_string: (distance, parent_string) }
        visited_f = {phi_a.to_str(): (0, None)}
        visited_b = {phi_b.to_str(): (0, None)}

        def reconstruct_path(meet_node_str):
            # 1. Reconstruct forward path (from start to meet_node)
            path_f = []
            curr = meet_node_str
            while curr is not None:
                path_f.append(curr)
                curr = visited_f[curr][1]  # get parent
            path_f.reverse()
            
            # 2. Reconstruct backward path (from meet_node to end)
            path_b = []
            curr = visited_b[meet_node_str][1]
            while curr is not None:
                path_b.append(curr)
                curr = visited_b[curr][1]  # get parent

            return path_f + path_b

        while queue_f and queue_b:
            # Expand forward
            curr_f, dist_f = queue_f.popleft()
            for neighbor in self.mutator.mutate(curr_f):
                n_str = neighbor.to_str()
                if n_str in visited_b:
                    visited_f[n_str] = (dist_f + 1, curr_f.to_str())
                    final_dist = dist_f + 1 + visited_b[n_str][0]
                    return final_dist, reconstruct_path(n_str)
                if n_str not in visited_f:
                    visited_f[n_str] = (dist_f + 1, curr_f.to_str())
                    queue_f.append((neighbor, dist_f + 1))

            # Expand backward
            curr_b, dist_b = queue_b.popleft()
            for neighbor in self.mutator.mutate(curr_b):
                n_str = neighbor.to_str()
                if n_str in visited_f:
                    visited_b[n_str] = (dist_b + 1, curr_b.to_str())
                    final_dist = dist_b + 1 + visited_f[n_str][0]
                    return final_dist, reconstruct_path(n_str)
                if n_str not in visited_b:
                    visited_b[n_str] = (dist_b + 1, curr_b.to_str())
                    queue_b.append((neighbor, dist_b + 1))
        
        return float('inf'), []

    def calculate_normalized_distance(self, phi_a_str, phi_b_str):
        phi_a = spot.formula(phi_a_str)
        phi_b = spot.formula(phi_b_str)
        
        # Unpack the raw_dist and ignore the path using '_'
        raw_dist, _ = self.calculate_raw_distance(phi_a, phi_b)
        
        if raw_dist == float('inf'): return float('inf')
        max_size = max(self._formula_size(phi_a), self._formula_size(phi_b))
        return raw_dist / max_size if max_size > 0 else 0.0

class LTLMutationDistance:
    def __init__(self, ap_list):
        """
        Initialize the distance calculator with the allowed atomic propositions.
        """
        self.mutator = LTLMutator(ap_list)
    def calculate_raw_distance(self, source, target):
        return min(self.calculate_distance(source, target), self.calculate_distance(target, source))


    def calculate_distance(self, source, target):
        """
        Calculates the minimum mutation distance from source formula to target formula.
        
        :param source: Starting LTL formula string (e.g., "G(p U q)")
        :param target: Destination LTL formula string (e.g., "p U !q")
        :return: (int, list) The distance, and the step-by-step path taken. Returns (inf, []) if unreachable.
        """
        src_formula = spot.formula(source).unabbreviate("ie")
        tgt_formula = spot.formula(target).unabbreviate("ie")
        
        src_str = src_formula.to_str()
        tgt_str = tgt_formula.to_str()
        
        if src_str == tgt_str:
            return 0, [src_str]
            
        # BFS Queue holds tuples of (current_formula, path_taken_as_list)
        queue = deque([(src_formula, [src_str])])
        
        # Visited set tracks string representations to prevent infinite loops
        visited = {src_str}
        
        while queue:
            current_formula, current_path = queue.popleft()
            
            # Generate all valid 1-point mutations from the current formula
            neighbors = self.mutator.mutate(current_formula)
            
            for neighbor in neighbors:
                neighbor_str = neighbor.to_str()
                
                if neighbor_str == tgt_str:
                    return len(current_path), current_path + [neighbor_str]
                    
                if neighbor_str not in visited:
                    visited.add(neighbor_str)
                    queue.append((neighbor, current_path + [neighbor_str]))
                    
        return float('inf'), [] # If no structural mutation path connects them


    def calculate_normalized_distance(self, phi_a_str, phi_b_str):
        phi_a = spot.formula(phi_a_str)
        phi_b = spot.formula(phi_b_str)
        
        # Unpack the raw_dist and ignore the path using '_'
        raw_dist, _ = self.calculate_raw_distance(phi_a, phi_b)
        if raw_dist == float('inf'): return float('inf')
        max_size = max(self._formula_size(phi_a), self._formula_size(phi_b))
        return raw_dist / max_size if max_size > 0 else 0.0


In [89]:
# 1. Define the pool of atomic propositions used in your system
ap_pool = ['p', 'q']

# 2. Instantiate the Mutator
mutator = LTLMutator(ap_pool)

# 3. Supply the base LTL formula you wish to mutate
original_formula = "GF(p U q)"
mutations = mutator.mutate(original_formula)

print(f"Original Formula: {original_formula}")
print(f"Total Unique Mutations Found: {len(mutations)}")
print("-" * 30)

# 4. Display the results
for i, m in enumerate(mutations):
    print(f"{i + 1:02d}: {m.to_str()}")

Original Formula: GF(p U q)
Total Unique Mutations Found: 45
------------------------------
01: !GF(p U q)
02: XGF(p U q)
03: FGF(p U q)
04: !F(p U q)
05: XF(p U q)
06: F(p U q)
07: G!F(p U q)
08: GXF(p U q)
09: G!(p U q)
10: GX(p U q)
11: G(p U q)
12: GF!(p U q)
13: GFX(p U q)
14: GFG(p U q)
15: GF(p R q)
16: GF(p W q)
17: GF(p & q)
18: GF(p | q)
19: GFp
20: GFq
21: GF(!p U q)
22: GF(Xp U q)
23: GF(Fp U q)
24: GF(Gp U q)
25: GF(p U !q)
26: GF(p U Xq)
27: GF(p U Fq)
28: GF(p U Gq)
29: G(q U F(p U q))
30: G(q W F(p U q))
31: G(q & F(p U q))
32: G(q | F(p U q))
33: G(p U F(p U q))
34: G(p W F(p U q))
35: G(p & F(p U q))
36: G(p | F(p U q))
37: q U GF(p U q)
38: q W GF(p U q)
39: q & GF(p U q)
40: q | GF(p U q)
41: p U GF(p U q)
42: p W GF(p U q)
43: p & GF(p U q)
44: p | GF(p U q)
45: FG(p U q)


In [90]:
# Define the AP pool
ap_pool = ['p', 'q']
dist_calculator = LTLMutationDistance(ap_pool)

# Define source and destination formulas
start = "GF(p U q)"
end = "p U !(q->p)"

distance, path = dist_calculator.calculate_raw_distance(start, end)

print(f"Source: {start}")
print(f"Target: {end}")
print(f"Mutation Distance: {distance}")
print("Optimal Mutation Path:")
print(" |-> ".join(path))

Source: GF(p U q)
Target: p U !(q->p)
Mutation Distance: 3
Optimal Mutation Path:
p U !(p | !q) |-> F(p U !(p | !q)) |-> GF(p U !(p | !q)) |-> GF(p U q)


In [66]:
f_node = spot.formula("!GF(p U q)")
f_node.to_str()

'!GF(p U q)'

In [ ]:
ap_pool = ['p', 'q']

# 2. Instantiate the Mutator
mutator = LTLMutator(ap_pool)

# 3. Supply the base LTL formula you wish to mutate
original_formula = "!GF(p U q)"
mutations = mutator.mutate(original_formula)

print(f"Original Formula: {original_formula}")
print(f"Total Unique Mutations Found: {len(mutations)}")
print("-" * 30)

# 4. Display the results
for i, m in enumerate(mutations):
    print(f"{i + 1:02d}: {m.to_str()}")

Original Formula: GF(p U q)
Total Unique Mutations Found: 44
------------------------------
01: !GF(p U q)
02: XGF(p U q)
03: FGF(p U q)
04: !F(p U q)
05: XF(p U q)
06: F(p U q)
07: G!F(p U q)
08: GXF(p U q)
09: G!(p U q)
10: GX(p U q)
11: G(p U q)
12: GF!(p U q)
13: GFX(p U q)
14: GFG(p U q)
15: GF(p R q)
16: GF(p W q)
17: GF(p & q)
18: GF(p | q)
19: GFp
20: GFq
21: GF(!p U q)
22: GF(Xp U q)
23: GF(Fp U q)
24: GF(Gp U q)
25: GF(p U !q)
26: GF(p U Xq)
27: GF(p U Fq)
28: GF(p U Gq)
29: G(q U F(p U q))
30: G(q W F(p U q))
31: G(q & F(p U q))
32: G(q | F(p U q))
33: G(p U F(p U q))
34: G(p W F(p U q))
35: G(p & F(p U q))
36: G(p | F(p U q))
37: q U GF(p U q)
38: q W GF(p U q)
39: q & GF(p U q)
40: q | GF(p U q)
41: p U GF(p U q)
42: p W GF(p U q)
43: p & GF(p U q)
44: p | GF(p U q)


In [ ]:
from collections import deque
import spot

class LTLTree:
    """A pure, un-optimized Abstract Syntax Tree (AST) for LTL formulas."""
    def __init__(self, value, children=None):
        self.value = value  # e.g., 'G', 'F', '!', 'U', '&', 'p', 'q'
        self.children = children if children is not None else []

    @classmethod
    def from_spot(cls, f):
        """Recursively parses a Spot formula into a pure text AST."""
        kind = f.kind()
        
        if f.is_tt(): return cls("true")
        if f.is_ff(): return cls("false")
        
        # Check if Atomic Proposition
        if kind == spot.op_ap: 
            return cls(f.to_str())
        
        # Handle Unary Operators
        if kind in [spot.op_Not, spot.op_X, spot.op_F, spot.op_G]:
            op_str = {spot.op_Not: '!', spot.op_X: 'X', spot.op_F: 'F', spot.op_G: 'G'}[kind]
            return cls(op_str, [cls.from_spot(f[0])])
            
        # Handle Binary/N-ary Operators
        if kind in [spot.op_U, spot.op_R, spot.op_W, spot.op_And, spot.op_Or]:
            op_str = {spot.op_U: 'U', spot.op_R: 'R', spot.op_W: 'W', spot.op_And: '&', spot.op_Or: '|'}[kind]
            children = list(f)
            
            # FIX HERE: Manually fold flat multi-operand structures (e.g., [a, b, c]) 
            # into right-nested binary structures (e.g., (a & (b & c)))
            if len(children) > 2 and kind in [spot.op_And, spot.op_Or]:
                # Start from the last element and build upwards
                right_child = cls.from_spot(children[-1])
                for child in reversed(children[1:-1]):
                    right_child = cls(op_str, [cls.from_spot(child), right_child])
                return cls(op_str, [cls.from_spot(children[0]), right_child])
                
            return cls(op_str, [cls.from_spot(children[0]), cls.from_spot(children[1])])
            
        raise ValueError(f"Unsupported formula component: {f.to_str()}")

    def to_str(self):
        """Converts the tree back to a cleanly parenthesized string format."""
        if not self.children:
            return self.value
        if len(self.children) == 1:
            # Unary operators: e.g., !GF(p U q) -> !(G(F(p U q)))
            child_str = self.children[0].to_str()
            if len(self.value) > 1 or self.value.isalpha() or child_str.startswith('('):
                return f"{self.value}{child_str}"
            return f"{self.value}({child_str})"
        else:
            # Binary operators
            return f"({self.children[0].to_str()} {self.value} {self.children[1].to_str()})"

    def copy(self):
        return LTLTree(self.value, [c.copy() for c in self.children])


class PureLTLMutator:
    def __init__(self, ap_list):
        self.aps = ap_list
        self.unary_ops = ['!', 'X', 'F', 'G']
        self.binary_ops = ['U', 'R', 'W', '&', '|']

    def mutate(self, tree):
        mutations = set()
        for mut_tree in self._mutate_recursive(tree, is_top_level=True):
            mutations.add(mut_tree.to_str())
        orig_str = tree.to_str()
        if orig_str in mutations:
            mutations.remove(orig_str)
        return list(mutations)

    def _mutate_recursive(self, node, is_top_level=True):
        # --- GENERAL CASES ---
        # Rule 5: Wrap in unary operator
        for op in self.unary_ops:
            yield LTLTree(op, [node.copy()])
            
        # Rule 1 & 6: Constants/Leaves (Only at leaf level or structurally allowed)
        if node.value == "true": yield LTLTree("false")
        if node.value == "false": yield LTLTree("true")
        if node.value in ["false"] + self.aps: yield LTLTree("true")
        if node.value in ["true"] + self.aps: yield LTLTree("false")
        
        if node.value in self.aps:
            for p in self.aps:
                if p != node.value: yield LTLTree(p)
                
        if not is_top_level or node.value in (self.aps + ["true", "false"]):
            for p in self.aps:
                yield LTLTree(p)

        # --- INDUCTIVE CASES ---
        if node.value in self.unary_ops:
            child = node.children[0]
            
            # 3(a) Change unary operator
            for op in self.unary_ops:
                if op != node.value: yield LTLTree(op, [child.copy()])
            # 3(b) Drop operator
            yield child.copy()
            # 3(c) Mutate child
            for mut_child in self._mutate_recursive(child, is_top_level=False):
                yield LTLTree(node.value, [mut_child])
            # 3(d) Append binary operator
            for p in self.aps:
                for op in ['U', 'W', '&', '|']:
                    yield LTLTree(op, [LTLTree(p), node.copy()])

        elif node.value in self.binary_ops:
            left, right = node.children[0], node.children[1]
            
            # 4(a) Change binary operator
            for op in self.binary_ops:
                if op != node.value: yield LTLTree(op, [left.copy(), right.copy()])
            # 4(b) Keep one child
            yield left.copy()
            yield right.copy()
            # 4(c) Mutate left child
            for mut_left in self._mutate_recursive(left, is_top_level=False):
                yield LTLTree(node.value, [mut_left, right.copy()])
            # 4(d) Mutate right child
            for mut_right in self._mutate_recursive(right, is_top_level=False):
                yield LTLTree(node.value, [left.copy(), mut_right])


class PureLTLDistanceCalculator:
    def __init__(self, ap_list):
        self.mutator = PureLTLMutator(ap_list)

    def distance(self, start_formula, end_formula):
        # Normalize target string structure via the exact same syntax tree format
        src_tree = LTLTree.from_spot(spot.formula(start_formula))
        tgt_tree = LTLTree.from_spot(spot.formula(end_formula))
        
        src_str = src_tree.to_str()
        tgt_str = tgt_tree.to_str()
        
        if src_str == tgt_str:
            return 0, [src_str]
            
        queue = deque([(src_tree, [src_str])])
        visited = {src_str}
        
        while queue:
            curr_tree, path = queue.popleft()
            neighbors = self.mutator.mutate(curr_tree)
            
            for n_str in neighbors:
                if n_str == tgt_str:
                    return len(path), path + [n_str]
                if n_str not in visited:
                    visited.add(n_str)
                    queue.append((LTLTree.from_spot(spot.formula(n_str)), path + [n_str]))
                    
        return float('inf'), []
    

calc = PureLTLDistanceCalculator(['p', 'q'])
# Note: "q->p" maps syntactically to "!q | p" under standard unabbreviation tree formats
dist, path = calc.distance("GF(p U q)", "p U !( !q | p )")

print(f"Mutation Distance: {dist}")
print("Strict Structural Path:")
for step in path:
    print(f" -> {step}")

Mutation Distance: 5
Strict Structural Path:
 -> GF(p U q)
 -> GF(false U q)
 -> (p U GFq)
 -> (p U G!(q))
 -> (p U G(p | !(q)))
 -> (p U !(p | !(q)))


In [1]:
import spot

# 1. Parse your LTL formula
formula = spot.formula("G(a -> X b)")

# 2. Translate the formula into a Büchi Automaton
aut = formula.translate()

# 3. Find a satisfying run (a lasso)
run = aut.accepting_run()

if run:
    # This prints the handle and the cycle of the lasso
    print("Found a satisfying lasso:", run)
else:
    print("No satisfying lasso exists.")

Found a satisfying lasso: Prefix:
Cycle:
  0
  |  !a



<function spot._impl.formula_F>